# 04 · ¿Por qué no aparece un efecto y qué implica?

**Objetivo:** Mostrar a directores y dueños si la competencia cercana cambia el precio de una tienda, como caso de competencia local para decisiones de expansión y de precio por tienda.

**Pregunta:** ¿Cuánto margen tienen las cadenas para fijar precio por tienda?

Importa porque si las sucursales de una cadena cobran lo mismo en toda la ciudad, la competencia de la esquina
no tiene por dónde reflejarse en el precio de anaquel. Se mide cuánto varía el precio entre sucursales del mismo formato
en la misma ciudad (`estadistica.same_chain_same_city`, tolerancia de 2%).

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import competencia as K
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
REPS = 500
foc = pd.read_csv(PROC / "tiendas_focales.csv")
ev = foc[foc.grupo.isin(K.EVALUABLES)].copy()
pts, st = K.load()
x = pts.merge(st[["establecimiento_id", "formato", "municipio_id"]], on="establecimiento_id").rename(columns={"sku_id": "sku"})
x["cadena_comercial"] = x.formato
fm = ["Walmart", "Bodega Aurrerá", "Walmart Express", "Hipermercado Soriana", "Mega Soriana"]  # Chedraui casi no tiene 2 sucursales en la misma ciudad
_, same = E.same_chain_same_city(x, fm)
display(same.round(1))
w = ev.copy()
w["resid"] = w.indice - w.groupby(["municipio_id", "formato"]).indice.transform("mean")
sd = (w.groupby("formato").resid.std() * 100).rename("desviación del índice entre sucursales (puntos %)")
display(sd.round(2).to_frame())

,celdas,igual_pct
cadena_comercial,,
Bodega Aurrerá,6181,80.4
Hipermercado Soriana,4812,58.6
Mega Soriana,1378,32.5
Walmart,7032,86.2
Walmart Express,3028,87.3


,desviación del índice entre sucursales (puntos %)
formato,
Bodega Aurrerá,0.11
Bodega Aurrerá Express,0.28
Hipermercado Soriana,0.30
Mega Soriana,0.22
Mercado Soriana,0.32
Soriana Express,0.00
Soriana Super,0.00
Walmart,0.41
Walmart Express,0.15


In [2]:
same.to_csv(PROC / "mismo_formato_misma_ciudad.csv", encoding="utf-8-sig")
F = "04_implicaciones"
reg("igual_walmart", same.loc["Walmart", "igual_pct"], fuente=F); reg("igual_bodega", same.loc["Bodega Aurrerá", "igual_pct"], fuente=F)
reg("igual_soriana", same.loc["Hipermercado Soriana", "igual_pct"], fuente=F); reg("igual_mega", same.loc["Mega Soriana", "igual_pct"], fuente=F)
display(Markdown(
    f"**Conclusión.** Las sucursales del mismo formato en la misma ciudad cobran el mismo precio (±2%) en {same.loc['Walmart', 'igual_pct']:.0f}% de los "
    f"productos en Walmart, {same.loc['Bodega Aurrerá', 'igual_pct']:.0f}% en Bodega Aurrerá y {same.loc['Hipermercado Soriana', 'igual_pct']:.0f}% en Hipermercado "
    "Soriana. Con precios casi iguales por ciudad, el competidor de la esquina tiene poco espacio para mover el precio de una sucursal."))

**Conclusión.** Las sucursales del mismo formato en la misma ciudad cobran el mismo precio (±2%) en 86% de los productos en Walmart, 80% en Bodega Aurrerá y 59% en Hipermercado Soriana. Con precios casi iguales por ciudad, el competidor de la esquina tiene poco espacio para mover el precio de una sucursal.